# Replaying experimental recordings with the FlyBody model

In this tutorial we replay a snippet of experimentally recorded fly walking on a ball using the **FlyBody** body model — published in [Vaxenburg et al. (2025)](https://www.nature.com/articles/s41586-025-09029-4) ([code](https://github.com/TuragaLab/flybody)) — from the Turaga lab, while retaining the FlyGym API for scene composition, simulation, and control.

This mirrors [Tutorial 2](2_replaying_experimental_recordings.ipynb), which replays a recording with the default NeuroMechFly model, but here we drive the FlyBody skeleton and additionally demonstrate its **tendon actuators**. The recording is the same dataset presented in the original NeuroMechFly publication; custom code based on [SeqIKPy](https://nely-epfl.github.io/sequential-inverse-kinematics/) was used to extract the kinematic chain and run the inverse kinematics. See [Tutorial 5b](5b_using_flybody_model.ipynb) for running a walking controller on the same model.

!!! warning "Experimental"
    Support for the FlyBody body model is **experimental**. The API may change in future releases, and not all features available for the default NeuroMechFly model are currently supported.


## Load the motion snippet

The data is bundled as a clip under `flygym_demo/ball_flybody_data/assets/ball_flybody_clip.npz`. It already matches the `MotionSnippet` schema — the same helper used in [Tutorial 2](2_replaying_experimental_recordings.ipynb) — so we can load it directly and move on to replaying the motion.

In [ ]:
from importlib.resources import files
from pathlib import Path

import numpy as np
from tqdm import trange

from flygym import Simulation
from flygym.compose import ActuatorType, FlatGroundWorld, KinematicPosePreset
from flygym.compose.fly import FlyBody
from flygym.flybody import (
    FlyBodyActuatedDOFPreset,
    FlyBodyAxisOrder,
    FlyBodyContactBodiesPreset,
    FlyBodyJointPreset,
    FlyBodySkeleton,
)
from flygym.utils.math import Rotation3D
from flygym_demo.spotlight_data import MotionSnippet

snippet_path = (
    Path(str(files("flygym_demo"))) / "ball_flybody_data/assets/ball_flybody_clip.npz"
)
snippet = MotionSnippet(snippet_path, angles_global2anatomical=False)

print("Experimental data and metadata:")
print(f"  legs: {snippet.legs}")
print(f"  dofs_per_leg: {snippet.dofs_per_leg}")
print(f"  data_fps: {snippet.data_fps}")
print(f"  experiment_trial: {snippet.experiment_trial}")
print(f"  framerange_in_raw_recording: {snippet.framerange_in_raw_recording}")
print(
    f"  joint_angles: shape={snippet.joint_angles.shape}, dtype={snippet.joint_angles.dtype}"
)
print(
    f"  fwdkin_egoxyz: shape={snippet.fwdkin_egoxyz.shape}, dtype={snippet.fwdkin_egoxyz.dtype}"
)
print(
    f"  rawpred_egoxyz: shape={snippet.rawpred_egoxyz.shape}, dtype={snippet.rawpred_egoxyz.dtype}"
)

## Build the FlyBody model

The model uses leg-only articulation with position-controlled active leg DoFs, plus leg adhesion and the FlyBody-specific **tendon actuators**. The active DOF layout (42 position actuators) matches the standard locomotion fly, but here it is built on the FlyBody skeleton, axis convention, and neutral pose.

In [ ]:
axis_order = FlyBodyAxisOrder.YAW_ROLL_PITCH
neutral_pose = KinematicPosePreset.FLYBODY_NEUTRAL
actuator_type = ActuatorType.POSITION

fly = FlyBody(name="flybody")
skeleton = FlyBodySkeleton(
    axis_order=axis_order,
    joint_preset=FlyBodyJointPreset.LEGS_ONLY,
)
fly.add_joints(skeleton, neutral_pose)

position_actuated_dofs = fly.skeleton.get_actuated_dofs_from_preset(
    FlyBodyActuatedDOFPreset.LEGS_ACTIVE_ONLY
)
fly.add_actuators(position_actuated_dofs, actuator_type, kp=100)
fly.add_tendons()
fly.add_tendon_actuators()
fly.add_leg_adhesion()
fly.colorize()

body_cam = fly.add_tracking_camera(
    name="body_cam",
    pos_offset=(0.0, -8.0, 0),
    rotation=Rotation3D("euler", (1.57, 0.0, 0.0)),
    fovy=35.0,
)

world = FlatGroundWorld()
world.add_fly(
    fly,
    [0, 0, 0.7],
    Rotation3D("quat", [1, 0, 0, 0]),
    bodysegs_with_ground_contact=FlyBodyContactBodiesPreset.LEGS_THORAX_ABDOMEN_HEAD,
    add_ground_contact_sensors=True,
)

sim = Simulation(world)
renderer = sim.set_renderer(
    [body_cam],
    camera_res=(240, 320),
    playback_speed=0.1,
    output_fps=25,
)

print("FlyBody model ready")
print("  position actuators:", len(position_actuated_dofs))
print("  tendon actuators:", len(fly.get_actuated_jointdofs_order(ActuatorType.TENDON)))

## Replay the recording

The target joint trajectories are obtained through `MotionSnippet.get_joint_angles`, which handles smoothing, interpolation, and DOF reordering for the simulator. During the replay itself we keep the tendon actuators neutral.

In [ ]:
sim_timestep = sim.timestep
position_targets = snippet.get_joint_angles(
    output_timestep=sim_timestep,
    output_dof_order=fly.get_actuated_jointdofs_order(actuator_type),
    sgfilter_window_sec=0.05,
)
print("Position targets shape:", position_targets.shape)

In [ ]:
nsteps_sim = position_targets.shape[0]
n_dofs = len(fly.get_jointdofs_order())
n_position_actuators = len(fly.get_actuated_jointdofs_order(actuator_type))
n_tendon_actuators = len(fly.get_actuated_jointdofs_order(ActuatorType.TENDON))

simulated_joint_angles = np.full((nsteps_sim, n_dofs), np.nan, dtype=np.float32)
position_actuator_forces = np.full(
    (nsteps_sim, n_position_actuators), np.nan, dtype=np.float32
)

fly_name = fly.name
zero_tendon_inputs = np.zeros(n_tendon_actuators, dtype=np.float32)

sim.reset()
sim.set_tendon_actuator_inputs(fly_name, zero_tendon_inputs)
sim.warmup()
smooth_start_time = 0.1
smooth_start_timesteps = int(smooth_start_time / sim_timestep)

for step_idx in trange(nsteps_sim, desc="Replaying"):
    target_angles = position_targets[step_idx, :] * min(
        1.0, step_idx / smooth_start_timesteps
    )
    sim.set_actuator_inputs(fly_name, actuator_type, target_angles)
    sim.step_with_profile()
    simulated_joint_angles[step_idx, :] = sim.get_joint_angles(fly_name)
    position_actuator_forces[step_idx, :] = sim.get_actuator_forces(
        fly_name, actuator_type
    )
    sim.render_as_needed_with_profile()

print("Replay complete")
print("  replay steps:", nsteps_sim)
print("  actuator target shape:", position_targets.shape)

## Show the replay

After the replay finishes, we can display the rendered video inline and inspect the runtime profile.

In [ ]:
sim.print_performance_report()
sim.renderer.show_in_notebook()

## Tendon actuation demo

The FlyBody model incorporates tendons to more faithfully actuate certain body parts. Here we build a fly with all biological joints — including the abdomen — and use a tendon actuator to drive the abdomen while holding the position actuators fixed.

In [ ]:
from flygym.compose import TetheredWorld

fly = FlyBody(name="flybody")
skeleton = FlyBodySkeleton(
    axis_order=axis_order,
    joint_preset=FlyBodyJointPreset.ALL_BIOLOGICAL,
)
fly.add_joints(skeleton, neutral_pose)

actuated_dofs = fly.skeleton.get_actuated_dofs_from_preset(FlyBodyActuatedDOFPreset.ALL)
fly.add_actuators(
    actuated_dofs,
    actuator_type=actuator_type,
    kp=1.0,  # just to silence the warning
)
fly.add_tendons()
fly.add_tendon_actuators()
fly.colorize()

tracking_cam = fly.add_tracking_camera(
    pos_offset=[-1, -5, 1],
    rotation=Rotation3D("xyaxes", (0.999, 0.044, -0.000, -0.012, 0.265, 0.964)),
)

world = TetheredWorld()
world.add_fly(
    fly,
    (0, 0, 0.0),
    Rotation3D("quat", (1, 0, 0, 0)),
)

sim = Simulation(world)
renderer = sim.set_renderer([tracking_cam])
fly_name = fly.name

In [ ]:
sim_timestep = sim.timestep
nsteps_tendon = int(2.0 / sim_timestep)
hold_position = np.zeros(len(actuated_dofs), dtype=np.float32)
n_tendon_actuators = len(fly.get_actuated_jointdofs_order(ActuatorType.TENDON))
tendon_drive = np.zeros((nsteps_tendon, n_tendon_actuators), dtype=np.float32)
if tendon_drive.shape[1] > 0:
    tendon_drive[:, 0] = 0.35 * np.sin(np.linspace(0, 8 * np.pi, nsteps_tendon))

sim.reset()
print("Tendon actuators:", n_tendon_actuators)

In [ ]:
for step_idx in trange(nsteps_tendon, desc="Tendon demo"):
    sim.set_actuator_inputs(fly_name, actuator_type, hold_position)
    sim.set_tendon_actuator_inputs(fly_name, tendon_drive[step_idx])
    sim.step_with_profile()
    sim.render_as_needed_with_profile()

sim.renderer.show_in_notebook()